# Processing a city with utca

In [ ]:
import utca
import osmnx as ox
import neatnet
import networkx as nx

## Load city street graph

In [ ]:
place_name = {"city": "Szombathely", "country": "Hungary"}
my_filter = utca.params.filter
crs = utca.params.crs

In [ ]:
G = ox.graph_from_place(place_name, custom_filter=my_filter)
G = ox.project_graph(G, to_crs=crs)
G = utca.prepare_graph(G)
poly_sequence = utca.polygonize(G)
polygons = utca.poly_df(G, poly_sequence)

We can visualize it on a map:

In [ ]:
utca.explore_graph(G, polygons)

And calculate various metrics:

In [ ]:
utca.graph_stats(G)

## Simplifying the graph

Removing roundabouts, simplifying complex intersections, removing false nodes, etc.

In [ ]:
streets = ox.graph_to_gdfs(G, node_geometry=False, nodes=False, edges=True)
neat = neatnet.neatify(streets)

Reconstruct the graph from the simplified `neat` geodataframe containing only streets:

In [ ]:
neat["length"] = neat.geometry.length
nodes, edges = utca.rebuild_neat_graph(neat)
G2 = ox.graph_from_gdfs(nodes, edges)
G2 = G2.to_undirected()
G2 = utca.prepare_graph(G2)
# ! largest cc ----
largest_cc = max(nx.connected_components(G2), key=len)
G2 = G2.subgraph(largest_cc).copy()
# ! -----
G2 = utca.remove_all_roundabouts(G2)
G2 = utca.prepare_graph(G2)

Visualize the simplified network:

In [ ]:
polygons2 = utca.poly_df(G2)
utca.explore_graph(G2, polygons2)

And the stats for the simiplified network:

In [ ]:
utca.graph_stats(G2)